# Experimento 007 - Treinamento real via API REST

Este notebook reproduz o escopo do Experimento 007: autenticar no TAO FTMS, validar o contexto e submeter um job real de treinamento `classification_pyt` com uma GPU e uma epoca.

Este notebook executa treinamento real por padrao. Configure `TAO_BASE_URL` para o endpoint do FTMS, `NGC_KEY` ou `TAO_NGC_KEY` para a chave NGC, e defina `RUN_EXPERIMENT_007=false` somente para validar o payload sem submeter um job. A chave deve estar em `TAO_SECRETS_FILE`, `secrets.json` ou no ambiente.

## Escopo

- Health e readiness do FTMS.
- Login NGC sem exibir a chave.
- Validacao do workspace, dataset e schema de treino.
- Treinamento real com `num_epochs=1` e `num_gpus=1`.
- Polling ate estado terminal e salvamento da evidencia JSON.

Este notebook nao executa avaliacao, exportacao, TensorRT ou inferencia; essas etapas pertencem ao pipeline completo do Experimento 001.

In [9]:
import copy
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import requests

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://100.107.81.126:8090").rstrip("/")
TAO_ORG = os.getenv("TAO_ORG", "getter")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
DATASET_URI = os.getenv("TAO_TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/tao-test-classification-v1")
BASE_EXPERIMENT_ID = os.getenv("TAO_BASE_EXPERIMENT_ID", "469f42fe-37dc-55ff-b89b-5096477f9a87")
HISTORICAL_BACKBONE = "fan_hybrid_tiny"
CORRECTED_BACKBONE = "fan_small_12_p4_hybrid"
RUN_HISTORICAL_CONTROL = True
RUN_CORRECTED_TRAINING = True
EVIDENCE_DIR = Path(os.getenv("TAO_EXPERIMENT_006_EVIDENCE", "experiment-006-evidence"))
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
configured = os.getenv("TAO_SECRETS_FILE", "")
candidates = [Path(configured)] if configured else []
candidates += [Path("secrets.json"), Path.cwd() / "secrets.json", Path.cwd().parent / "secrets.json", Path("/home/getter/tao-ftms/secrets.json")]
secrets_file = next((item for item in candidates if item.exists()), None)
NGC_KEY = json.loads(secrets_file.read_text())["ngc_api_key"] if secrets_file else os.getenv("NGC_KEY", "")
if not NGC_KEY: raise RuntimeError("NGC key not found")
http = requests.Session()
print({"base_url": TAO_BASE_URL, "historical": HISTORICAL_BACKBONE, "corrected": CORRECTED_BACKBONE, "historical_control": RUN_HISTORICAL_CONTROL, "corrected_control": RUN_CORRECTED_TRAINING})

{'base_url': 'http://100.107.81.126:8090', 'historical': 'fan_hybrid_tiny', 'corrected': 'fan_small_12_p4_hybrid', 'historical_control': True, 'corrected_control': True}


In [10]:
def api(method, path, payload=None, headers=None, timeout=(5, 30)):
    merged = {"Content-Type": "application/json"}
    if headers:
        merged.update(headers)
    response = http.request(method, TAO_BASE_URL + path, headers=merged, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json() if response.content else None

print("login: starting")
login = api("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
TOKEN = login["token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
print("login: PASS")
health = api("GET", "/api/v2/health")
workspace = api("GET", f"/api/v2/orgs/{TAO_ORG}/workspaces/{WORKSPACE_ID}", headers=AUTH)
datasets = api("GET", f"/api/v2/orgs/{TAO_ORG}/datasets", headers=AUTH)
schema_response = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:schema?network_arch={NETWORK_ARCH}&action=train", headers=AUTH)
catalog = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:list_base_experiments?network_arch={NETWORK_ARCH}", headers=AUTH)

assert "readiness" in health
assert workspace.get("id") == WORKSPACE_ID
assert isinstance(schema_response, dict)
print("context=PASS datasets=", len(datasets.get("datasets", [])), "base_experiments=", len(catalog.get("experiments", [])))

login: starting
login: PASS
context=PASS datasets= 1 base_experiments= 26


## Geracao do payload a partir do schema real

O notebook nao usa um payload fixo. Ele parte do `default` retornado pelo TAO e altera somente os parametros necessarios para o experimento.

In [11]:
train_specs = copy.deepcopy(schema_response.get("default", schema_response))
train_specs.setdefault("dataset", {}).update({"root_dir": DATASET_URI, "num_classes": int(os.getenv("TAO_NUM_CLASSES", "2"))})
train_specs.setdefault("train", {}).update({"num_gpus": 1, "num_epochs": 1})

payload = {
    "kind": "experiment",
    "name": f"experiment-007-training-{uuid.uuid4().hex[:12]}",
    "network_arch": NETWORK_ARCH,
    "encryption_key": os.getenv("TAO_ENCRYPTION_KEY", "tlt_encode"),
    "workspace": WORKSPACE_ID,
    "action": "train",
    "specs": train_specs,
    "train_dataset_uris": [DATASET_URI],
    "eval_dataset_uri": os.getenv("TAO_EVAL_DATASET_URI", DATASET_URI),
    "inference_dataset_uri": os.getenv("TAO_INFERENCE_DATASET_URI", DATASET_URI),
    "base_experiment_ids": [BASE_EXPERIMENT_ID],
}
(EVIDENCE_DIR / "train-payload.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print({"name": payload["name"], "epochs": train_specs["train"]["num_epochs"], "gpus": train_specs["train"]["num_gpus"], "dataset": DATASET_URI})

{'name': 'experiment-007-training-305bd01d861b', 'epochs': 1, 'gpus': 1, 'dataset': 'seaweedfs://tao-storage/data/tao-test-classification-v1'}


## Submissao e polling

A celula abaixo submete o job real quando `RUN_EXPERIMENT_007` esta habilitado e aguarda o estado terminal `Done`. Use `RUN_EXPERIMENT_007=false` apenas para dry-run.

In [12]:
if not RUN_EXPERIMENT:
    print("dry-run: RUN_EXPERIMENT_007=false; nenhum job foi submetido")
else:
    submitted = api("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
    job_id = submitted["id"]
    (EVIDENCE_DIR / "train-submitted.json").write_text(json.dumps(submitted, indent=2), encoding="utf-8")
    terminal = {"Done", "Completed", "Error", "Failed", "Failure", "Canceled", "Cancelled"}
    deadline = time.monotonic() + int(os.getenv("TAO_JOB_TIMEOUT_SECONDS", str(TIMEOUT_SECONDS)))
    while time.monotonic() < deadline:
        current = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{job_id}", headers=AUTH)
        (EVIDENCE_DIR / "train-status.json").write_text(json.dumps(current, indent=2), encoding="utf-8")
        state = current.get("status")
        print("training", state)
        if state in terminal:
            break
        time.sleep(POLL_SECONDS)
    else:
        raise TimeoutError(job_id)
    if current.get("status") not in {"Done", "Completed"}:
        raise RuntimeError(json.dumps(current, indent=2))
    print(json.dumps({"job_id": job_id, "status": current.get("status")}, indent=2))

training Pending
training Pending
training Started
training Started
training Started
training Started
training Done
{
  "job_id": "7e9adbfc-b47e-4d74-8b6b-1d4bcbda9505",
  "status": "Done"
}


In [13]:
summary = {
    "experiment": "007",
    "scope": "real training via TAO REST API",
    "base_url": TAO_BASE_URL,
    "network_arch": NETWORK_ARCH,
    "workspace_id": WORKSPACE_ID,
    "dataset_uri": DATASET_URI,
    "run_enabled": RUN_EXPERIMENT,
    "evidence_dir": str(EVIDENCE_DIR),
}
print(json.dumps(summary, indent=2))

{
  "experiment": "007",
  "scope": "real training via TAO REST API",
  "base_url": "http://100.107.81.126:8090",
  "network_arch": "classification_pyt",
  "workspace_id": "98c4bec1-88df-423e-8a15-af78881e9742",
  "dataset_uri": "seaweedfs://tao-storage/data/tao-test-classification-v1",
  "run_enabled": true,
  "evidence_dir": "experiment-006-evidence"
}
